# Day 24 — Project: Headcount and Salary Analytics
**Estimated time:** 90 minutes  
**Learning objectives:**
1. Build HCM-style workforce metrics: headcount, tenure, new hires/terminations, turnover
2. Perform salary band analysis by job title and org unit
3. Identify cost centers where headcount cost exceeds budget by joining HR to CO data

---
> **SAP Context:** HR data in SAP HCM uses PERNR (personnel number), PLANS (position), STELL (job title), ORGEH (org unit), KOSTL (cost center). Termination date is stored in PA0000 (action infotype). SALARY maps roughly to IT0008 (basic pay). This analysis mirrors what you'd build from a PA/OM extract for a workforce analytics dashboard.


## 0. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

pd.set_option('display.float_format', '{:,.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

from analytics_bootcamp.config import RAW_DATA_DIR as DATA
TODAY = pd.Timestamp('2026-04-01')

hr = pd.read_csv(DATA / 'hr_headcount.csv', parse_dates=['HIRE_DATE', 'TERM_DATE'])
cc = pd.read_csv(DATA / 'cost_center_actuals.csv')

print('HR shape:', hr.shape)
print('Active employees:', hr['TERM_DATE'].isna().sum())
hr.head(3)

## 1. Active Headcount by Org Unit + Tenure Distribution

In [ ]:
# YOUR SOLUTION
# Active = TERM_DATE is NaT OR TERM_DATE > TODAY
active = hr[(hr['TERM_DATE'].isna()) | (hr['TERM_DATE'] > TODAY)].copy()

headcount_by_org = active.groupby('ORGTX')['PERNR'].count().reset_index()
headcount_by_org.columns = ['ORG_UNIT', 'HEADCOUNT']
headcount_by_org = headcount_by_org.sort_values('HEADCOUNT', ascending=False)
print('=== Active Headcount by Org Unit ===')
print(headcount_by_org.to_string(index=False))

# Tenure in years
active = active.copy()
active['TENURE_YEARS'] = (TODAY - active['HIRE_DATE']).dt.days / 365.25

bins = [0, 1, 3, 5, 10, 100]
labels = ['<1yr', '1-3yr', '3-5yr', '5-10yr', '10+yr']
active['TENURE_BAND'] = pd.cut(active['TENURE_YEARS'], bins=bins, labels=labels)
print('\n=== Tenure Distribution ===')
print(active['TENURE_BAND'].value_counts().sort_index())

## 2. New Hires vs Terminations by Month

In [ ]:
# YOUR SOLUTION
# New hires: group by hire month
hr['HIRE_MONTH'] = hr['HIRE_DATE'].dt.to_period('M')
new_hires = hr.groupby('HIRE_MONTH')['PERNR'].count().reset_index()
new_hires.columns = ['MONTH', 'NEW_HIRES']

# Terminations: group by term month (only termed employees)
termed = hr[hr['TERM_DATE'].notna()].copy()
termed['TERM_MONTH'] = termed['TERM_DATE'].dt.to_period('M')
terminations = termed.groupby('TERM_MONTH')['PERNR'].count().reset_index()
terminations.columns = ['MONTH', 'TERMINATIONS']

# Merge
movement = pd.merge(new_hires, terminations, on='MONTH', how='outer').fillna(0)
movement = movement.sort_values('MONTH')
movement['NET_CHANGE'] = movement['NEW_HIRES'] - movement['TERMINATIONS']

# Show last 12 months
print(movement.tail(12).to_string(index=False))

## 3. Salary Band Analysis (Median, 25th/75th Percentile by Job Title and Org)

In [ ]:
# YOUR SOLUTION
salary_bands = (
    active.groupby(['STELL', 'ORGTX'])['SALARY']
    .agg(
        COUNT='count',
        MEDIAN=lambda x: x.median(),
        P25=lambda x: x.quantile(0.25),
        P75=lambda x: x.quantile(0.75),
        MIN='min',
        MAX='max'
    )
    .reset_index()
    .sort_values('MEDIAN', ascending=False)
)

print('=== Salary Bands by Job Title and Org Unit ===')
print(salary_bands.head(15).to_string(index=False))

## 4. Turnover Rate Calculation (Annualized)

In [ ]:
# YOUR SOLUTION
# Annualized turnover = (terminations in last 12m / avg headcount) * 100
last_12m_start = TODAY - pd.DateOffset(months=12)

terms_12m = hr[
    (hr['TERM_DATE'].notna()) &
    (hr['TERM_DATE'] >= last_12m_start) &
    (hr['TERM_DATE'] <= TODAY)
].shape[0]

# Average headcount = (headcount at start + headcount at end) / 2
hc_end = hr[(hr['TERM_DATE'].isna()) | (hr['TERM_DATE'] > TODAY)].shape[0]
hc_start = hr[(hr['HIRE_DATE'] <= last_12m_start) &
              ((hr['TERM_DATE'].isna()) | (hr['TERM_DATE'] > last_12m_start))].shape[0]
avg_hc = (hc_start + hc_end) / 2

turnover_rate = (terms_12m / avg_hc) * 100

print(f'Terminations (last 12m): {terms_12m}')
print(f'Avg Headcount: {avg_hc:.1f}')
print(f'Annualized Turnover Rate: {turnover_rate:.1f}%')

# Turnover by org unit
def turnover_by_org(hr_df, start, end):
    results = []
    for org in hr_df['ORGTX'].unique():
        subset = hr_df[hr_df['ORGTX'] == org]
        t = subset[(subset['TERM_DATE'].notna()) &
                   (subset['TERM_DATE'] >= start) & (subset['TERM_DATE'] <= end)].shape[0]
        hc = subset[(subset['HIRE_DATE'] <= end) &
                    ((subset['TERM_DATE'].isna()) | (subset['TERM_DATE'] > end))].shape[0]
        rate = (t / hc * 100) if hc > 0 else 0
        results.append({'ORG': org, 'TERMINATIONS': t, 'END_HC': hc, 'TURNOVER_RATE_PCT': round(rate, 1)})
    return pd.DataFrame(results).sort_values('TURNOVER_RATE_PCT', ascending=False)

org_turnover = turnover_by_org(hr, last_12m_start, TODAY)
print('\n=== Turnover Rate by Org Unit ===')
print(org_turnover.to_string(index=False))

## 5. Identify Cost Centers Where Headcount Cost Exceeds Budget

In [ ]:
# YOUR SOLUTION
# Salary spend per cost center (active employees only)
cc_salary = (
    active.groupby('KOSTL')['SALARY']
    .sum()
    .reset_index()
    .rename(columns={'SALARY': 'ANNUAL_SALARY_COST'})
)

# Compare to plan (YTD salaries from cost_center_actuals, cost element 400000 = Salaries)
cc_salary_budget = (
    cc[cc['KSTAR'] == 400000]
    .groupby('KOSTL')
    .agg(PLAN_SALARY=('PLAN_AMT', 'sum'), ACTUAL_SALARY=('ACTUAL_AMT', 'sum'))
    .reset_index()
)

# Join
combined = cc_salary.merge(cc_salary_budget, on='KOSTL', how='inner')
combined['HEADCOUNT_VS_PLAN'] = combined['ANNUAL_SALARY_COST'] - combined['PLAN_SALARY']
combined['OVERAGE_PCT'] = (combined['HEADCOUNT_VS_PLAN'] / combined['PLAN_SALARY'] * 100).round(1)

over_budget_cc = combined[combined['HEADCOUNT_VS_PLAN'] > 0].sort_values('HEADCOUNT_VS_PLAN', ascending=False)
print('=== Cost Centers: Headcount Cost vs Salary Budget ===')
print(over_budget_cc.to_string(index=False))

## 6. Workforce Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Headcount by Org Unit
hc_top10 = headcount_by_org.head(10)
hc_top10.plot(kind='barh', x='ORG_UNIT', y='HEADCOUNT', ax=axes[0],
              color='#3498db', edgecolor='white', legend=False)
axes[0].set_title('Active Headcount by Org Unit (Top 10)', fontweight='bold')
axes[0].set_xlabel('Headcount')
axes[0].invert_yaxis()

# Chart 2: Tenure Distribution
tenure_counts = active['TENURE_BAND'].value_counts().sort_index()
tenure_counts.plot(kind='bar', ax=axes[1], color='#9b59b6', edgecolor='white')
axes[1].set_title('Tenure Distribution', fontweight='bold')
axes[1].set_ylabel('Employee Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('day24_workforce_charts.png', dpi=120, bbox_inches='tight')
plt.show()

## Daily Prompt
**Answer independently:**

1. The CHRO asks for a "regrettable attrition" report — terminations where the employee had tenure > 5 years. Filter and count these employees, and calculate their average salary.
2. In SAP HCM, what table would you query to get an employee's position history (not just current position)? Why does position history matter for headcount analytics?
3. Extend the salary band analysis to include a "compa-ratio" column: `SALARY / MEDIAN` for each employee. Flag anyone below 0.8 (underpaid risk) or above 1.2 (overpaid risk).
